# PROD Data Analizi

Bu notebookta .csv dosyası olarak alınan input dosyası için:
- FILE_SIZE ve NUMBER_OF_RECORDS sütunlarının ilişkisi incelenir.
- 3 farklı kategorideki veriler için boyut/trend analizi yapılır.
- REGEX_ERROR_COUNT değişkeni kullanılarak error sayısının trend analizi yapılır.

FILE_SIZE ve NUMBER_OF_RECORDS sütunlarının orantısına göre analiz sırasında hangi değişkenin dosya boyutu olarak kullanılacağı kararlaştırılır, analiz sonuçlarına (düzenli trend sayısı, ani iniş veya çıkışların varlığı) bakılarak validasyon için kullanılacak olan yapay zeka modeli seçilir.

In [ ]:
import pandas as pd # veri manipulasyonu
import numpy as np # sayisal islemler
import matplotlib.pyplot as plt # gorsellestirme

# dataframe display ayarlari
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Data Yüklemesi

Input dosyası .csv formatındadır ve aşağıdaki sütunları içermesi beklenir: 

    ID, FILE_TP_ID, FILE_NAME, FILE_STATUS,	FILE_SIZE, NUMBER_OF_RECORDS,	REGEX_ERROR_COUNT

Dosyasının eksik olup olmaması ve sütunları hakkında varsayım yapılmaz, dosya kontrollerden geçirilir.

In [ ]:
CSV_PATH = "/kaggle/input/datasets/aycaatac/prod-data/import_file_output.csv"


In [ ]:
df_base = pd.read_csv(CSV_PATH) # .csv dosyasi data frame olarak yuklenir
df_base.shape # satir ve sutun sayisi kontrol


In [ ]:
# icerisinde ',' bulunan sayisal veriler ',' karakterinden arindirilir
numeric_cols = ['ID', 'FILE_SIZE', 'NUMBER_OF_RECORDS']

for col in numeric_cols:
    df_base[col] = pd.to_numeric(
        df_base[col].astype(str).str.replace(',', ''), 
        errors='coerce' # parse hatalarinda deger NaN'e cevrilir
    )

In [ ]:
# data icerik ve formati kontrol edilir
df_base.head()


## Tablo Kontrolü

Parsing/mapping hatalarını önlemek için sütun tipleri ve içerikleri kontrol edilir. Tüm kayıtların veya yalnızca successful kayıtların analizinin yapılması için bir değişken tanımlanır. 

In [ ]:
# data tipleri kontrol edilir
df_base.dtypes


In [ ]:
# sayisal olmasi gereken degerler sayisala cevrilir ve file status alani temizlenir
df_base["FILE_TP_ID"] = pd.to_numeric(df_base["FILE_TP_ID"], errors="coerce")
df_base["FILE_SIZE"] = pd.to_numeric(df_base["FILE_SIZE"], errors="coerce")
df_base["NUMBER_OF_RECORDS"] = pd.to_numeric(df_base["NUMBER_OF_RECORDS"], errors="coerce")
df_base["REGEX_ERROR_COUNT"] = pd.to_numeric(df_base["REGEX_ERROR_COUNT"], errors="coerce")
df_base["FILE_STATUS"] = df_base["FILE_STATUS"].astype(str).str.upper().str.strip()
df_base.shape


In [ ]:
# sadece successful kayitlarin kullanilmasi icin bir variable
USE_ONLY_TRAINING_DATA = False


In [ ]:
# kayitlar icerisinden successful ve sonu .gz ile bitmeyen satirlar secilir
df = df_base.copy()

if USE_ONLY_TRAINING_DATA:
    df = df[
        (df["FILE_STATUS"] == "S") &
        (~df["FILE_NAME"].str.endswith(".gz", na=False))
    ]


## Eksik Data Analizi

Eksik data analizi:

- içerisinde null değerler bulunduran alanları tespit etmek,
- modellemeye uygun olmayan alanları tespit etmek,
- data kalitesini incelemek
  
için yapılır.

In [ ]:
# sutunlar NaN degerler icin kontrol edilir, her sutundaki NaN orani kontrol edilir
df.isna().mean().sort_values(ascending=False)


## Kategori ve File Status Kontrolü

In [ ]:
# dosya ve kategori tipleri teyit edilir
df["FILE_TP_ID"].value_counts().sort_index()


## FILE_SIZE ve NUMBER_OF_RECORDS İlişkisi 

Dosya boyutu değerlendirilirken hangi sütunun kullanılacağına karar vermek için iki sütunun tamamen doğru orantı içerisinde olup olmadığı kontrol edilir. Dosya boyutunu değerlendirmek için kullanılacak olan değişken tanımlanır ve FILE_SIZE veya NUMBER_OF_RECORDS'a eşitlenir.

In [ ]:
# her dosya kategorisi icin NUMBER_OF_RECORDS ve FILE_SIZE arasındaki korelasyon hesaplanir
df.groupby("FILE_TP_ID")[["NUMBER_OF_RECORDS","FILE_SIZE"]].corr()


In [ ]:
# global korelasyon kontrol edilir
df[["NUMBER_OF_RECORDS","FILE_SIZE"]].corr()


In [ ]:
# hangi degiskenin dosya boyutuna bakmak icin kullanilacagina karar verilir
VOLUME_COL = "NUMBER_OF_RECORDS"
df["VOLUME_SIGNAL"] = df[VOLUME_COL]


## Kategorilere Göre Data Analizi

Her kategori için ayrı ayrı trend analizi yapılır.

In [ ]:
# var olan kategoriler cikartilir
categories = sorted(df["FILE_TP_ID"].dropna().unique())
categories


In [ ]:
# her kategori icin histogram cizilir - x: NUMBER_OF_RECORDS y: frequency (kac dosya o aralikta)
for tp in categories:
    subset = df[df["FILE_TP_ID"] == tp]
    plt.figure(figsize=(6,4))
    plt.hist(subset["VOLUME_SIGNAL"].dropna(), bins=50)
    plt.title(f"{VOLUME_COL} distribution | FILE_TP_ID={tp}")
    plt.show()


In [ ]:
# her kategori icin dosya sayisi, ortalama record sayisi, standart sapma vs. hesaplanir
df.groupby("FILE_TP_ID")["VOLUME_SIGNAL"].describe()


In [ ]:
# verilerin zaman boyunca nasil degistigini gozlemlemek icin scatter plot olusturulur
# x: verinin zaman sirasi, y: NUMBER_OF_RECORDS
for tp in categories:
    subset = df[df["FILE_TP_ID"] == tp]
    plt.figure(figsize=(6,4))
    plt.scatter(
        subset.index,
        subset["VOLUME_SIGNAL"],
        alpha=0.5
    )
    plt.title(f"{VOLUME_COL} scatter | FILE_TP_ID={tp}")
    plt.show()


In [ ]:
# her kategori icin NUMBER_OF_RECORDS kullanilarak boxplot olusturulur
# distributioni ve global threshold olusturulup olusturulamayacagini gosterir
for tp in categories:
    subset = df[df["FILE_TP_ID"] == tp]
    plt.figure(figsize=(6,4))
    plt.boxplot(subset["VOLUME_SIGNAL"].dropna(), vert=False)
    plt.title(f"{VOLUME_COL} boxplot | FILE_TP_ID={tp}")
    plt.show()

In [ ]:
# FILE_NAME sutunundan FILE_TIMESTAMP ve FILE_DATE sutunu olusturulur
df["FILE_TIMESTAMP"] = pd.to_datetime(
    df["FILE_NAME"].str.extract(r"(\d{14})")[0],
    format="%Y%m%d%H%M%S"
)

df = df.sort_values("FILE_TIMESTAMP")
df["FILE_DATE"] = df["FILE_TIMESTAMP"].dt.date


# kayitlar FILE_DATE'e gore sortlanir
df = df.sort_values("FILE_DATE")

In [ ]:
# LEVEL icin NUMBER_OF_RECORDS ve zaman grafigi olusturulur
for tp in categories:
    subset = df[df["FILE_TP_ID"] == tp]
    plt.figure(figsize=(10,4))
    plt.plot(subset["FILE_DATE"], subset["VOLUME_SIGNAL"])
    plt.title(f"{VOLUME_COL} over time | FILE_TP_ID={tp}")
    plt.xlabel("Date")
    plt.ylabel(VOLUME_COL)
    plt.show()

In [ ]:
window = 30 # son 30 gun incelenir

# son 30 gunun ortalama ve oynakligina bakilir
for tp in categories:
    subset = df[df["FILE_TP_ID"] == tp].set_index("FILE_DATE")

    plt.figure(figsize=(10,4))
    plt.plot(subset["VOLUME_SIGNAL"], alpha=0.4, label="Daily")
    plt.plot(subset["VOLUME_SIGNAL"].rolling(window).mean(), label="Rolling Mean")
    plt.plot(subset["VOLUME_SIGNAL"].rolling(window).std(), label="Rolling Std")

    plt.title(f"Rolling stats ({window} days) | FILE_TP_ID={tp}")
    plt.legend()
    plt.show()

In [ ]:
# Z-score normalization uygulanir ve olcekler esitlenir
df_norm = df.copy()

df_norm["VOLUME_NORM"] = (
    df_norm.groupby("FILE_TP_ID")["VOLUME_SIGNAL"]
    .transform(lambda x: (x - x.mean()) / x.std())
)


In [ ]:
# kategorilerdeki normalize edilmis degerlerin ayni olcekte karsilastirilmasi x: date y: kategorinin 
# +- std arttirilmis degerleri
plt.figure(figsize=(12,5))

for tp in categories:
    subset = df_norm[df_norm["FILE_TP_ID"] == tp]
    plt.plot(
        subset["FILE_DATE"],
        subset["VOLUME_NORM"],
        label=f"FILE_TP_ID={tp}"
    )

plt.title("Normalized Volume Signal Over Time (All Categories)")
plt.xlabel("Date")
plt.ylabel("Normalized Volume")
plt.legend()
plt.show()


In [ ]:
# haftalik ortalama ve trend analizi
for tp in categories:
    subset = df[df["FILE_TP_ID"] == tp].set_index("FILE_DATE")
    weekly = subset["VOLUME_SIGNAL"].resample("W").mean()

    plt.figure(figsize=(10,4))
    plt.plot(weekly)
    plt.title(f"Weekly mean volume | FILE_TP_ID={tp}")
    plt.show()


## REGEX_ERROR_COUNT Analizi

Error sayısı için trend analizi yapılır. Kategoriler ve dosya büyüklüğü ile error sayısı orantısı incelenir.

In [ ]:
# kategorilere gore REGEX_ERROR_COUNT incelemesi, x: error sayısı, y: bu error sayısına sahip olan
# dosya sayısı
for tp in categories:
    subset = df[df["FILE_TP_ID"] == tp]
    plt.figure(figsize=(6,4))
    plt.hist(subset["REGEX_ERROR_COUNT"].dropna(), bins=50)
    plt.title(f"REGEX_ERROR_COUNT distribution | FILE_TP_ID={tp}")
    plt.show()

In [ ]:
# her kategorideki dosyalarin yuzde kacinin en az bir hata icerdigine bakilir
df["HAS_REGEX_ERROR"] = df["REGEX_ERROR_COUNT"].fillna(0) > 0
df.groupby("FILE_TP_ID")["HAS_REGEX_ERROR"].mean()

In [ ]:
# dosya boyutu ve hata sayisi arasindaki iliski kontrol edilir
for tp in categories:
    subset = df[df["FILE_TP_ID"] == tp]
    plt.figure(figsize=(6,4))
    plt.scatter(
        subset["VOLUME_SIGNAL"],
        subset["REGEX_ERROR_COUNT"],
        alpha=0.5
    )
    plt.title(f"{VOLUME_COL} vs REGEX_ERROR_COUNT | FILE_TP_ID={tp}")
    plt.show()

In [ ]:
# her kategori icin dosya buyuklugu ve error sayisi arasindaki Pearson korelasyonu hesaplanir
df.groupby("FILE_TP_ID")[["VOLUME_SIGNAL", "REGEX_ERROR_COUNT"]].corr()


In [ ]:
# error sayısı ve error rate ile dosya boyutu arasındaki korelasyonlar kontrol edilir
t3 = df[df["FILE_TP_ID"] == 3].copy()

t3["ERROR_RATE"] = (
    t3["REGEX_ERROR_COUNT"] /
    t3["VOLUME_SIGNAL"]
)

t3[["VOLUME_SIGNAL","REGEX_ERROR_COUNT","ERROR_RATE"]].corr(method="pearson")
t3[["VOLUME_SIGNAL","REGEX_ERROR_COUNT","ERROR_RATE"]].corr(method="spearman")


In [ ]:
# her kategori icin kayit ve ortalama hata sayisi, std, min/max, ceyrek kontrolleri yapilir
df.groupby(["FILE_TP_ID","FILE_STATUS"])["REGEX_ERROR_COUNT"].describe()


In [ ]:
# error count ve file status arasindaki iliski kontrol edilir
pd.crosstab(
    df["HAS_REGEX_ERROR"],
    df["FILE_STATUS"],
    normalize="index"
)


In [ ]:
df.groupby("FILE_TP_ID").agg(
    mean_volume=("VOLUME_SIGNAL","mean"),
    std_volume=("VOLUME_SIGNAL","std"),
    mean_errors=("REGEX_ERROR_COUNT","mean"),
    error_rate=("HAS_REGEX_ERROR","mean")
)


In [ ]:
# 2 ve 3. kategorilerin zaman icerisindeki error sayisi gorsellestirilir
for tp in [2, 3]:
    subset = df[df["FILE_TP_ID"] == tp].set_index("FILE_DATE")
    
    plt.figure(figsize=(10,4))
    plt.plot(subset["REGEX_ERROR_COUNT"])
    plt.title(f"Daily REGEX_ERROR_COUNT | FILE_TP_ID={tp}")
    plt.ylabel("Error Count")
    plt.show()


In [ ]:
# 3. kategoride bir trend olmasi ihtimaline karsi error sayilari aylara gore incelenir
t3["YEAR_MONTH"] = t3["FILE_DATE"].dt.to_period("M")
t3.groupby("YEAR_MONTH")["REGEX_ERROR_COUNT"].mean()



## Model Seçimi için Detaylı Analiz

In [ ]:
df = df.sort_values(["FILE_TP_ID", "FILE_DATE", "ID"])
df["FILE_ORDER"] = df.groupby(["FILE_TP_ID", "FILE_DATE"]).cumcount()

print(df["FILE_ORDER"].value_counts().sort_index())

print(df.groupby(["FILE_TP_ID", "FILE_ORDER"])["VOLUME_SIGNAL"].describe())

print(df.groupby(["FILE_TP_ID", "FILE_DATE"])["FILE_ORDER"].nunique().value_counts().sort_index())


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["VOLUME_SIGNAL"].mean()
    
    plt.figure(figsize=(8,4))
    plot_acf(series.dropna(), lags=60)
    plt.title(f"ACF | FILE_TP_ID {tp}")
    plt.show()


In [ ]:
df = df.sort_values(["FILE_TP_ID", "PROCESSED_DATE"])

df["DELTA"] = (
    df.groupby("FILE_TP_ID")["VOLUME_SIGNAL"]
      .diff()
)

from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp]
    series = subset.groupby("FILE_DATE")["DELTA"].mean()
    
    plt.figure(figsize=(8,4))
    plot_acf(series.dropna(), lags=60)
    plt.title(f"ACF (DELTA) | FILE_TP_ID {tp}")
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["DELTA"].mean()
    
    plt.figure(figsize=(12,4))
    plt.plot(series)
    plt.title(f"DELTA over time | FILE_TP_ID {tp}")
    plt.show()


In [ ]:
df["WEEKDAY"] = df["FILE_DATE"].dt.weekday

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp]
    weekday_means = subset.groupby("WEEKDAY")["DELTA"].mean()
    
    print(f"\nFILE_TP_ID {tp}")
    print(weekday_means)


In [ ]:
df["YEAR_MONTH"] = df["FILE_DATE"].dt.to_period("M")

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp]
    month_means = subset.groupby("YEAR_MONTH")["DELTA"].mean()
    
    print(f"\nFILE_TP_ID {tp}")
    print(month_means)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import skew, kurtosis

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp]
    series = subset["DELTA"].dropna()
    
    print(f"\nFILE_TP_ID {tp}")
    print("Skew:", skew(series))
    print("Kurtosis:", kurtosis(series))
    
    plt.figure(figsize=(6,4))
    sns.histplot(series, bins=100, kde=True)
    plt.title(f"DELTA Distribution | FILE_TP_ID {tp}")
    plt.show()


In [ ]:
for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["DELTA"].mean()
    
    rolling_std = series.rolling(60).std()
    
    plt.figure(figsize=(10,4))
    plt.plot(rolling_std)
    plt.title(f"Rolling 60-day STD | FILE_TP_ID {tp}")
    plt.show()


In [ ]:
for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["DELTA"].mean()
    
    rolling_median = series.rolling(60).median()
    
    plt.figure(figsize=(10,4))
    plt.plot(rolling_median)
    plt.title(f"Rolling 60-day Median | FILE_TP_ID {tp}")
    plt.show()


In [ ]:
for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["DELTA"].mean()
    
    median = series.median()
    mad = np.median(np.abs(series - median))
    threshold = 3 * 1.4826 * mad
    
    outliers = (np.abs(series - median) > threshold).astype(int)
    rolling_outlier_rate = outliers.rolling(60).mean()
    
    plt.figure(figsize=(10,4))
    plt.plot(rolling_outlier_rate)
    plt.title(f"Rolling Outlier Rate | FILE_TP_ID {tp}")
    plt.show()


In [ ]:
for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["DELTA"].mean()
    
    split_date = "2025-01-01"
    before = series[series.index < split_date]
    after = series[series.index >= split_date]
    
    print(f"\nFILE_TP_ID {tp}")
    print("STD before 2025:", before.std())
    print("STD after 2025:", after.std())


In [ ]:
df["processed_date"] = pd.to_datetime(
    df["PROCESSED_DATE"],
    format="%d/%m/%Y %H:%M:%S,%f"
)

df["create_date"] = pd.to_datetime(
    df["CREATE_DATE"],
    format="%d/%m/%Y %H:%M:%S,%f"
)


In [ ]:
batch_counts = df.groupby("processed_date")["FILE_TP_ID"].nunique()

batch_counts.value_counts().sort_index()


In [ ]:
df = df.sort_values("processed_date")

df["time_diff_sec"] = df["processed_date"].diff().dt.total_seconds()

df[["FILE_TP_ID", "processed_date", "time_diff_sec"]].head(20)


In [ ]:
import pandas as pd
import numpy as np

print("rows:", len(df))
print("cols:", list(df.columns))

for c in ["FILE_TP_ID", "FILE_STATUS", "FILE_SIZE", "PROCESSED_DATE", "CREATE_DATE"]:
    print(c, "exists:", c in df.columns)

print("\nFILE_STATUS counts:")
print(df["FILE_STATUS"].value_counts(dropna=False).head(20))

def _to_number(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(".", "").replace(",", "")
    try:
        return float(s)
    except:
        return np.nan

tmp = df.copy()
tmp["FILE_SIZE_NUM"] = tmp["FILE_SIZE"].map(_to_number)

print("\nFILE_SIZE_NUM missing ratio:", tmp["FILE_SIZE_NUM"].isna().mean())
print("FILE_SIZE_NUM basic stats:")
print(tmp["FILE_SIZE_NUM"].describe(percentiles=[.01,.05,.5,.95,.99]))

print("\nRecords per FILE_TP_ID:")
print(tmp["FILE_TP_ID"].value_counts().sort_index())


In [ ]:
print("FILE_STATUS dağılımı:")
print(df["FILE_STATUS"].value_counts(dropna=False))

print("\nFILE_STATUS = 'F' kayıt sayısı:")
print((df["FILE_STATUS"] == "F").sum())


In [ ]:
failed = df[df["FILE_STATUS"] == "F"].copy()
failed["ERROR_DESCRIPTION"].value_counts()


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox


print("="*70)
print("PART 1: WEEKLY SEASONALITY TEST (VOLUME_SIGNAL levels, not DELTA)")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].copy()
    subset["WEEKDAY"] = subset["FILE_DATE"].dt.weekday
    
    weekday_means = subset.groupby("WEEKDAY")["VOLUME_SIGNAL"].mean()
    weekday_std = subset.groupby("WEEKDAY")["VOLUME_SIGNAL"].std()
    overall_mean = subset["VOLUME_SIGNAL"].mean()
    overall_std = subset["VOLUME_SIGNAL"].std()
    
    print(f"\n{'='*50}")
    print(f"FILE_TP_ID = {tp}")
    print(f"{'='*50}")
    print(f"\nOverall mean: {overall_mean:,.0f}")
    print(f"Overall std:  {overall_std:,.0f}")
    print(f"\nWeekday means:")
    
    weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    for i, (mean, std) in enumerate(zip(weekday_means, weekday_std)):
        pct_diff = (mean - overall_mean) / overall_mean * 100
        print(f"  {weekday_names[i]}: {mean:>15,.0f}  ({pct_diff:+.2f}% from overall)")
    
    groups = [subset[subset["WEEKDAY"] == i]["VOLUME_SIGNAL"].values for i in range(7)]
    groups = [g for g in groups if len(g) > 0] 
    
    if len(groups) >= 2:
        f_stat, p_value = stats.f_oneway(*groups)
        print(f"\nANOVA test (weekday effect):")
        print(f"  F-statistic: {f_stat:.2f}")
        print(f"  p-value: {p_value:.4f}")
        if p_value < 0.05:
            print(f"  >>> SIGNIFICANT weekday effect detected (p < 0.05)")
        else:
            print(f"  >>> NO significant weekday effect (p >= 0.05)")

print("\n")
print("="*70)
print("PART 2: ACF VALUES AT KEY LAGS")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["VOLUME_SIGNAL"].mean()
    
    acf_values = acf(series.dropna(), nlags=35, fft=True)
    
    print(f"\nFILE_TP_ID = {tp}")
    print(f"  Lag 1 (yesterday):  {acf_values[1]:.3f}")
    print(f"  Lag 7 (weekly):     {acf_values[7]:.3f}")
    print(f"  Lag 14 (2 weeks):   {acf_values[14]:.3f}")
    print(f"  Lag 30 (monthly):   {acf_values[30]:.3f}")
    
    n = len(series)
    sig_threshold = 1.96 / np.sqrt(n)
    print(f"  95% significance threshold: ±{sig_threshold:.3f}")
    
    if abs(acf_values[7]) > sig_threshold:
        print(f"  >>> SIGNIFICANT weekly autocorrelation")
    else:
        print(f"  >>> NO significant weekly autocorrelation")

print("\n")
print("="*70)
print("PART 3: LJUNG-BOX TEST FOR AUTOCORRELATION")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["VOLUME_SIGNAL"].mean().dropna()
    
    lb_result = acorr_ljungbox(series, lags=[7, 14, 30], return_df=True)
    
    print(f"\nFILE_TP_ID = {tp}")
    print(lb_result)
    
    if lb_result["lb_pvalue"].iloc[0] < 0.05:
        print(f"  >>> SIGNIFICANT autocorrelation up to lag 7")
    else:
        print(f"  >>> NO significant autocorrelation up to lag 7")

print("\n")
print("="*70)
print("PART 4: STL DECOMPOSITION (if enough data)")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["VOLUME_SIGNAL"].mean()
    
    if len(series) < 14:
        print(f"\nFILE_TP_ID = {tp}: Not enough data for STL")
        continue
    
    try:
        stl = STL(series, period=7, robust=True)
        result = stl.fit()
        
        var_resid = np.var(result.resid)
        var_seasonal = np.var(result.seasonal)
        var_total = np.var(series)
        
        seasonal_strength = max(0, 1 - var_resid / (var_resid + var_seasonal))
        trend_strength = max(0, 1 - var_resid / (var_resid + np.var(result.trend)))
        
        print(f"\nFILE_TP_ID = {tp}")
        print(f"  Seasonal strength (weekly): {seasonal_strength:.3f}")
        print(f"  Trend strength: {trend_strength:.3f}")
        print(f"  (Values > 0.6 indicate strong pattern)")
        
        if seasonal_strength > 0.6:
            print(f"  >>> STRONG weekly seasonality")
        elif seasonal_strength > 0.3:
            print(f"  >>> MODERATE weekly seasonality")
        else:
            print(f"  >>> WEAK/NO weekly seasonality")
            
        fig, axes = plt.subplots(4, 1, figsize=(12, 10))
        result.observed.plot(ax=axes[0], title=f"FILE_TP_ID {tp} - Observed")
        result.trend.plot(ax=axes[1], title="Trend")
        result.seasonal.plot(ax=axes[2], title="Seasonal (weekly)")
        result.resid.plot(ax=axes[3], title="Residual")
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"\nFILE_TP_ID = {tp}: STL failed - {e}")

print("\n")
print("="*70)
print("PART 5: COEFFICIENT OF VARIATION BY WEEKDAY")
print("="*70)
print("(If CV is similar across weekdays, weekday doesn't matter)")

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].copy()
    subset["WEEKDAY"] = subset["FILE_DATE"].dt.weekday
    
    cv_by_weekday = subset.groupby("WEEKDAY")["VOLUME_SIGNAL"].apply(
        lambda x: x.std() / x.mean() * 100
    )
    
    print(f"\nFILE_TP_ID = {tp}")
    print("Coefficient of Variation by weekday:")
    weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    for i, cv in enumerate(cv_by_weekday):
        print(f"  {weekday_names[i]}: {cv:.2f}%")
    
    cv_range = cv_by_weekday.max() - cv_by_weekday.min()
    print(f"  Range: {cv_range:.2f}%")
    if cv_range < 2:
        print(f"  >>> Weekday has MINIMAL effect on variability")

print("\n")
print("="*70)
print("SUMMARY")
print("="*70)
print("""
To determine if seasonality/trend exists, look for:

1. ANOVA p-value < 0.05 → Weekday matters
2. ACF at lag 7 > threshold → Weekly pattern
3. Ljung-Box p-value < 0.05 → Autocorrelation exists
4. STL seasonal strength > 0.6 → Strong seasonality
5. STL trend strength > 0.6 → Strong trend

If NONE of these are significant → Use adaptive Median/MAD
If SOME are significant → Consider hybrid approach
If MOST are significant → STL/MSTL may help
""")

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(series, period=7)
res = stl.fit()

detrended = series - res.trend


In [ ]:
from scipy.signal import periodogram
import numpy as np

freqs, power = periodogram(detrended)

plt.figure(figsize=(10,4))
plt.plot(freqs, power)
plt.title("Full Frequency Spectrum")
plt.show()


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(detrended, lags=120)
plt.show()


In [ ]:
df["PROCESSED_DATE"] = pd.to_datetime(df["PROCESSED_DATE"])

df["weekday"] = df["PROCESSED_DATE"].dt.weekday

df.groupby("weekday")["DELTA"].std()


In [ ]:
df["PROCESSED_DATE"] = pd.to_datetime(df["PROCESSED_DATE"])

df["day"] = df["PROCESSED_DATE"].dt.day

df["is_end"] = df["day"] >= 25

df.groupby("is_end")["DELTA"].mean()


In [ ]:
!pip install ruptures


In [ ]:
import ruptures as rpt

algo = rpt.Pelt(model="rbf").fit(detrended.values)
result = algo.predict(pen=10)

result


In [ ]:
import numpy as np
import statsmodels.api as sm

t3 = df[df["FILE_TP_ID"] == 3].copy()

series = t3["DELTA"].dropna().values

In [ ]:
t = np.arange(len(series))

In [ ]:
sin_term = np.sin(2 * np.pi * t / 7)
cos_term = np.cos(2 * np.pi * t / 7)

X = np.column_stack([sin_term, cos_term])
X = sm.add_constant(X)   

In [ ]:
model = sm.OLS(series, X).fit()

print(model.summary())

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.tsa.stattools import acf

print("="*70)
print("TEST 1: MONTHLY SEASONALITY (Day of Month)")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].copy()
    subset["DAY_OF_MONTH"] = subset["FILE_DATE"].dt.day
    
    subset["MONTH_PERIOD"] = pd.cut(subset["DAY_OF_MONTH"], 
                                     bins=[0, 10, 20, 31], 
                                     labels=["start", "mid", "end"])
    
    period_means = subset.groupby("MONTH_PERIOD")["VOLUME_SIGNAL"].mean()
    overall_mean = subset["VOLUME_SIGNAL"].mean()
    
    print(f"\nFILE_TP_ID = {tp}")
    for period, mean in period_means.items():
        pct_diff = (mean - overall_mean) / overall_mean * 100
        print(f"  {period}: {mean:,.0f} ({pct_diff:+.2f}%)")
    
    groups = [subset[subset["MONTH_PERIOD"] == p]["VOLUME_SIGNAL"].values 
              for p in ["start", "mid", "end"]]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"  ANOVA p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("  >>> SIGNIFICANT month-period effect")
    else:
        print("  >>> NO significant month-period effect")


print("\n")
print("="*70)
print("TEST 2: CALENDAR MONTH SEASONALITY (Jan vs Feb vs Mar...)")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].copy()
    subset["MONTH"] = subset["FILE_DATE"].dt.month
    
    month_means = subset.groupby("MONTH")["VOLUME_SIGNAL"].mean()
    overall_mean = subset["VOLUME_SIGNAL"].mean()
    
    print(f"\nFILE_TP_ID = {tp}")
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", 
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    for month, mean in month_means.items():
        pct_diff = (mean - overall_mean) / overall_mean * 100
        print(f"  {month_names[month-1]}: {mean:,.0f} ({pct_diff:+.2f}%)")
    
    groups = [subset[subset["MONTH"] == m]["VOLUME_SIGNAL"].values 
              for m in subset["MONTH"].unique()]
    groups = [g for g in groups if len(g) > 5]  # need enough samples
    
    if len(groups) >= 2:
        f_stat, p_value = stats.f_oneway(*groups)
        print(f"  ANOVA p-value: {p_value:.4f}")
        if p_value < 0.05:
            print("  >>> SIGNIFICANT calendar month effect")
        else:
            print("  >>> NO significant calendar month effect")


print("\n")
print("="*70)
print("TEST 3: QUARTER SEASONALITY (Q1 vs Q2 vs Q3 vs Q4)")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].copy()
    subset["QUARTER"] = subset["FILE_DATE"].dt.quarter
    
    quarter_means = subset.groupby("QUARTER")["VOLUME_SIGNAL"].mean()
    overall_mean = subset["VOLUME_SIGNAL"].mean()
    
    print(f"\nFILE_TP_ID = {tp}")
    for q, mean in quarter_means.items():
        pct_diff = (mean - overall_mean) / overall_mean * 100
        print(f"  Q{q}: {mean:,.0f} ({pct_diff:+.2f}%)")
    
    groups = [subset[subset["QUARTER"] == q]["VOLUME_SIGNAL"].values 
              for q in subset["QUARTER"].unique()]
    groups = [g for g in groups if len(g) > 5]
    
    if len(groups) >= 2:
        f_stat, p_value = stats.f_oneway(*groups)
        print(f"  ANOVA p-value: {p_value:.4f}")
        if p_value < 0.05:
            print("  >>> SIGNIFICANT quarter effect")
        else:
            print("  >>> NO significant quarter effect")


print("\n")
print("="*70)
print("TEST 4: ACF AT ALL KEY LAGS")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["VOLUME_SIGNAL"].mean()
    
    diff_series = series.diff().dropna()
    
    max_lag = min(120, len(diff_series) // 3)
    acf_values = acf(diff_series, nlags=max_lag, fft=True)
    
    n = len(diff_series)
    sig_threshold = 1.96 / np.sqrt(n)
    
    print(f"\nFILE_TP_ID = {tp} (differenced to remove trend)")
    print(f"  95% threshold: ±{sig_threshold:.3f}")
    print(f"  Lag 7 (weekly):     {acf_values[7]:.3f} {'*' if abs(acf_values[7]) > sig_threshold else ''}")
    print(f"  Lag 14 (bi-weekly): {acf_values[14]:.3f} {'*' if abs(acf_values[14]) > sig_threshold else ''}")
    print(f"  Lag 30 (monthly):   {acf_values[30]:.3f} {'*' if abs(acf_values[30]) > sig_threshold else ''}")
    
    if max_lag >= 90:
        print(f"  Lag 90 (quarterly): {acf_values[90]:.3f} {'*' if abs(acf_values[90]) > sig_threshold else ''}")
    
    significant_lags = [i for i in range(2, max_lag) if abs(acf_values[i]) > sig_threshold]
    if significant_lags:
        print(f"  Significant lags: {significant_lags[:10]}...") 
    else:
        print("  No significant lags found")


print("\n")
print("="*70)
print("TEST 5: END-OF-MONTH EFFECT")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].copy()
    subset["DAY_OF_MONTH"] = subset["FILE_DATE"].dt.day
    subset["IS_END_OF_MONTH"] = subset["DAY_OF_MONTH"] >= 25
    
    means = subset.groupby("IS_END_OF_MONTH")["VOLUME_SIGNAL"].mean()
    
    print(f"\nFILE_TP_ID = {tp}")
    print(f"  Days 1-24:  {means[False]:,.0f}")
    print(f"  Days 25-31: {means[True]:,.0f}")
    pct_diff = (means[True] - means[False]) / means[False] * 100
    print(f"  Difference: {pct_diff:+.2f}%")
    
    group1 = subset[~subset["IS_END_OF_MONTH"]]["VOLUME_SIGNAL"]
    group2 = subset[subset["IS_END_OF_MONTH"]]["VOLUME_SIGNAL"]
    t_stat, p_value = stats.ttest_ind(group1, group2)
    print(f"  T-test p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("  >>> SIGNIFICANT end-of-month effect")
    else:
        print("  >>> NO significant end-of-month effect")


print("\n")
print("="*70)
print("TEST 6: BEGINNING-OF-MONTH EFFECT")
print("="*70)

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].copy()
    subset["DAY_OF_MONTH"] = subset["FILE_DATE"].dt.day
    subset["IS_START_OF_MONTH"] = subset["DAY_OF_MONTH"] <= 5
    
    means = subset.groupby("IS_START_OF_MONTH")["VOLUME_SIGNAL"].mean()
    
    print(f"\nFILE_TP_ID = {tp}")
    print(f"  Days 1-5:   {means[True]:,.0f}")
    print(f"  Days 6-31:  {means[False]:,.0f}")
    pct_diff = (means[True] - means[False]) / means[False] * 100
    print(f"  Difference: {pct_diff:+.2f}%")
    
    group1 = subset[subset["IS_START_OF_MONTH"]]["VOLUME_SIGNAL"]
    group2 = subset[~subset["IS_START_OF_MONTH"]]["VOLUME_SIGNAL"]
    t_stat, p_value = stats.ttest_ind(group1, group2)
    print(f"  T-test p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("  >>> SIGNIFICANT start-of-month effect")
    else:
        print("  >>> NO significant start-of-month effect")

In [ ]:
for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].copy()
    subset["YEAR"] = subset["FILE_DATE"].dt.year
    subset["MONTH"] = subset["FILE_DATE"].dt.month
    
    jan_by_year = subset[subset["MONTH"] == 1].groupby("YEAR")["VOLUME_SIGNAL"].mean()
    
    print(f"\nFILE_TP_ID = {tp}")
    print("January by year:")
    print(jan_by_year)
    
    if len(jan_by_year) >= 2:
        pct_diff = (jan_by_year.iloc[-1] - jan_by_year.iloc[0]) / jan_by_year.iloc[0] * 100
        print(f"Change: {pct_diff:+.1f}%")
        if abs(pct_diff) > 5:
            print(">>> NOT similar - confirms NO seasonality (just trend)")
        else:
            print(">>> Similar - might be seasonality")

In [ ]:
import numpy as np
from scipy.signal import periodogram
import matplotlib.pyplot as plt

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    
    series = subset.groupby("FILE_DATE")["VOLUME_SIGNAL"].mean()
    series = series.diff().dropna()
    
    freqs, power = periodogram(series)
    
    plt.figure(figsize=(8,4))
    plt.plot(freqs, power)
    plt.title(f"Periodogram | FILE_TP_ID {tp}")
    plt.xlabel("Frequency")
    plt.ylabel("Power")
    plt.show()


In [ ]:
import statsmodels.api as sm
import numpy as np

for tp in sorted(df["FILE_TP_ID"].unique()):
    subset = df[df["FILE_TP_ID"] == tp].sort_values("FILE_DATE")
    series = subset.groupby("FILE_DATE")["VOLUME_SIGNAL"].mean()
    
    y = series.diff().dropna() 
    
    t = np.arange(len(y))
    
    sin_w = np.sin(2*np.pi*t/7)
    cos_w = np.cos(2*np.pi*t/7)
    
    sin_a = np.sin(2*np.pi*t/365)
    cos_a = np.cos(2*np.pi*t/365)
    
    X = np.column_stack([sin_w, cos_w, sin_a, cos_a])
    X = sm.add_constant(X)
    
    model = sm.OLS(y, X).fit()
    
    print("\n===================================")
    print(f"FILE_TP_ID = {tp}")
    print(model.summary())


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model = SARIMAX(series, order=(1,1,1), seasonal_order=(1,1,1,7))
results = model.fit()
print(results.summary())
